In [6]:
import pandas as pd

# CSVの読み込み
aoi_latency = pd.read_csv("./exported_csv/fixation_sliced_by_time/AOI_latency_all_trials.csv")
final_selected = pd.read_csv("./exported_csv/final_selected_count.csv")


# 条件と紐づく trialリスト
condition_trials = {
    "不気味": ["1-1", "1-7", "2-5", "2-7", "3-5", "3-8"], 
    "中間": ["1-2", "1-8", "2-1", "2-8", "3-2", "3-7"],
    "自然": ["1-4", "1-5", "2-2", "2-4", "3-1", "3-4"],
    "実写真": ["1-3", "1-6", "2-3", "2-6", "3-3", "3-6"],
}

# AOI_latencyに一意ID列を作る
aoi_latency["key"] = (
    aoi_latency["subject_id"].astype(str) + "-" +
    aoi_latency["experiment_id"].astype(str) + "-" +
    aoi_latency["trial"].astype(str)
)

# final_selectedにもkeyを作る
final_keys = (
    final_selected["subject_id"].astype(str) + "-" +
    final_selected["experiment_id"].astype(str) + "-" +
    final_selected["trial"].astype(str)
)

# final_selectedに含まれるものだけ抽出
aoi_latency_filtered = aoi_latency[aoi_latency["key"].isin(final_keys)]

# 結果を格納する辞書（平均と件数）
mean_results = {}
count_results = {}

# 条件ごとに集計
for condition, id_list in condition_trials.items():
    # experiment_id-trialの文字列
    mask = (
        aoi_latency_filtered["experiment_id"].astype(str) + "-" +
        aoi_latency_filtered["trial"].astype(str)
    )
    # 条件に該当する行
    subset = aoi_latency_filtered[mask.isin(id_list)]
    # 被験者ごとの平均
    mean_per_subject = subset.groupby("subject_id")["aoi_latency_sec"].mean()
    count_per_subject = subset.groupby("subject_id")["aoi_latency_sec"].count()
    mean_results[condition] = mean_per_subject
    count_results[condition] = count_per_subject

# 被験者IDのユニークリスト
all_subjects = sorted(aoi_latency_filtered["subject_id"].unique())

# 平均値のDataFrame
mean_df = pd.DataFrame(index=all_subjects)
for cond, series in mean_results.items():
    mean_df[cond] = series
mean_df = mean_df[["実写真", "自然", "中間", "不気味"]]  # 列順を指定

# 件数（分母）のDataFrame
count_df = pd.DataFrame(index=all_subjects)
for cond, series in count_results.items():
    count_df[cond] = series
count_df = count_df[["実写真", "自然", "中間", "不気味"]]  # 列順を指定

# 表示確認
print("平均値:\n", mean_df)
print("件数:\n", count_df)

# CSVに保存
mean_df.to_csv("./exported_csv/fixation_sliced_by_time/mean_latency_per_condition.csv",index=True,header=True,encoding="utf-8-sig")
count_df.to_csv("./exported_csv/fixation_sliced_by_time/count_per_condition.csv",index=True,header=True,encoding="utf-8-sig")

平均値:
          実写真        自然         中間       不気味
1   1.991800  2.022400   2.327667  1.184000
2   2.851333  1.656667   2.996750  1.908250
4   2.296000  2.553833   2.686000  0.772400
5   0.000000       NaN   3.934000  0.000000
6   6.876500  5.293500   3.318000  3.147250
7        NaN       NaN   0.508000       NaN
9   2.643000  0.000000   0.000000  0.000000
11  1.024000  1.584000   0.363000  0.331000
12  9.589750  7.597750   1.625000  3.260333
13  1.671833  0.805833   1.740500  0.690800
15  2.658833  2.602167   2.164250  1.319200
16  5.925333  5.834500  14.361250  6.260800
17  1.422000  1.960750   2.598667  1.328000
19  4.104000  6.214167   3.410000  3.156200
件数:
     実写真   自然  中間  不気味
1   5.0  5.0   3  3.0
2   6.0  6.0   4  4.0
4   6.0  6.0   4  5.0
5   1.0  NaN   1  1.0
6   4.0  4.0   2  4.0
7   NaN  NaN   1  NaN
9   2.0  2.0   1  2.0
11  1.0  1.0   1  1.0
12  4.0  4.0   3  3.0
13  6.0  6.0   4  5.0
15  6.0  6.0   4  5.0
16  6.0  6.0   4  5.0
17  4.0  4.0   3  3.0
19  6.0  6.0   4  5.0